In [4]:
import pandas as pd
import numpy as np
import urllib
import os
import joblib

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("All ML libraries loaded!")

All ML libraries loaded!


In [5]:
server = r"DEVIL\SQLEXPRESS"
database = "ecommerce"

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + params,
    pool_pre_ping=True
)

print("SQL Server connected!")

SQL Server connected!


In [6]:
test_df = pd.read_sql(
    """
    SELECT TOP 10 *
    FROM dbo.Customer_Churn_ML
    """,
    engine
)

print("Rows:", len(test_df))
display(test_df)

c:\Users\sahuj\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Rows: 10


,customer_unique_id,recency,frequency,monetary,avg_order_value,customer_lifetime_days,churn_target
0,0000f46a3911fa3c0805444483337064,357,1,86.220001,86.220001,0,1
1,0000f6ccb0745a6a4b88665a16c9f078,141,1,43.619999,43.619999,0,1
2,0004aac84e0df4da2b147fca70cf8255,108,1,196.889999,196.889999,0,1
3,00053a61a98854899e70ed204dd4bafe,2,1,419.179993,419.179993,0,1
4,0005e1862207bf6ccc02e4228effd9a0,363,1,150.119995,150.119995,0,1
5,0006fdc98a402fceb4eb0ee528f6a8d4,227,1,29.000000,29.000000,0,1
6,00082cbe03e478190aadbea78542e933,103,1,126.260002,126.260002,0,1
7,000a5ad9c4601d2bbdd9ed765d5213b3,203,1,91.279999,91.279999,0,1
8,000bfa1d2f1a41876493be685390d6d3,154,1,46.849998,46.849998,0,1
9,000c8bdb58a29e7115cfc257230fb21b,80,1,29.000000,29.000000,0,1


In [7]:
# Load the full machine learning dataset into 'df'
df = pd.read_sql(
    """
    SELECT * 
    FROM dbo.Customer_Churn_ML
    """,
    engine
)

print(f"Full dataset loaded! Shape: {df.shape}")

Full dataset loaded! Shape: (55906, 7)


In [8]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(
    df["churn_target"].value_counts(
        dropna=False
    )
)

Columns:
['customer_unique_id', 'recency', 'frequency', 'monetary', 'avg_order_value', 'customer_lifetime_days', 'churn_target']

Missing Values:
customer_unique_id        0
recency                   0
frequency                 0
monetary                  0
avg_order_value           0
customer_lifetime_days    0
churn_target              0
dtype: int64

Target Distribution:
churn_target
1    55251
0      655
Name: count, dtype: int64


In [9]:
df = df.dropna(
    subset=["churn_target"]
).copy()

df["churn_target"] = (
    df["churn_target"]
    .astype(int)
)

X = df.drop(
    columns=[
        "customer_unique_id",
        "churn_target"
    ],
    errors="ignore"
)

y = df["churn_target"]

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

print("Features:", X.columns.tolist())
print("X Shape:", X.shape)
print("Y Shape:", y.shape)

Features: ['recency', 'frequency', 'monetary', 'avg_order_value', 'customer_lifetime_days']
X Shape: (55906, 5)
Y Shape: (55906,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (44724, 5)
Test: (11182, 5)


In [11]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = (
    negative / positive
    if positive > 0
    else 1
)

models = {

    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print("Models created!")

Models created!


In [12]:
trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(
        X_train,
        y_train
    )

    trained_models[name] = model

print("All models trained!")

Training Logistic Regression...
Training Random Forest...
Training XGBoost...
All models trained!


In [13]:
results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(
        X_test
    )[:, 1]

    results.append({

        "Model": name,

        "Accuracy": accuracy_score(
            y_test,
            y_pred
        ),

        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "ROC_AUC": roc_auc_score(
            y_test,
            y_prob
        )
    })

results_df = pd.DataFrame(
    results
).sort_values(
    "ROC_AUC",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
2,XGBoost,0.749687,0.989210,0.754954,0.856351,0.575792
0,Logistic Regression,0.579503,0.989514,0.580671,0.731866,0.553933
1,Random Forest,0.932660,0.988149,0.943173,0.965137,0.511252


In [14]:
for name, model in trained_models.items():

    predictions = model.predict(
        X_test
    )

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )


Logistic Regression
              precision    recall  f1-score   support

           0       0.01      0.48      0.03       131
           1       0.99      0.58      0.73     11051

    accuracy                           0.58     11182
   macro avg       0.50      0.53      0.38     11182
weighted avg       0.98      0.58      0.72     11182


Random Forest
              precision    recall  f1-score   support

           0       0.01      0.05      0.02       131
           1       0.99      0.94      0.97     11051

    accuracy                           0.93     11182
   macro avg       0.50      0.49      0.49     11182
weighted avg       0.98      0.93      0.95     11182


XGBoost
              precision    recall  f1-score   support

           0       0.01      0.31      0.03       131
           1       0.99      0.75      0.86     11051

    accuracy                           0.75     11182
   macro avg       0.50      0.53      0.44     11182
weighted avg       0.98      

In [15]:
best_model_name = results_df.iloc[0]["Model"]

best_model = trained_models[
    best_model_name
]

print(
    "BEST MODEL:",
    best_model_name
)


BEST MODEL: XGBoost


In [16]:
os.makedirs(
    "../models",
    exist_ok=True
)

joblib.dump(
    best_model,
    "../models/best_model.pkl"
)

print(
    "Best model saved successfully!"
)

Best model saved successfully!
